In [ ]:
! pip install -q -U bitsandbytes accelerate

In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

In [2]:
import pandas as pd 

file_path = '/kaggle/input/clusters/clusters.csv'

try:
    df_clusters = pd.read_csv(file_path, sep=',', encoding='cp1252', on_bad_lines='warn')
    print("Dataset loaded successfully! Here are the first few rows:")
    display(df_clusters.head())
except FileNotFoundError:
    print("File not found! Please double check the file_path.")

Dataset loaded successfully! Here are the first few rows:


,phrase,cluster
0,"located at second St NE, Uhland Terrace NE, Wa...",18
1,monthly rental rates range from $790 - $1090,21
2,studio units available for rent,20
3,"814 Schutte Road, Evansville, 47712, IN",5
4,Monthly rental rates range from $425 - $445,14


In [ ]:
!pip install -U transformers tokenizers huggingface_hub accelerate

In [3]:
import torch
from transformers import pipeline

2025-09-21 13:03:18.737866: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758459798.761198     235 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758459798.768302     235 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
text_gen_pipeline = pipeline(
    task="text-generation",
    model="google/gemma-3-4b-it",
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "quantization_config": {"load_in_4bit": True}
    }
)

print("Pipeline with quantized model loaded successfully!")

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Pipeline with quantized model loaded successfully!


In [ ]:
import random 

cluster_phrases_full = df_clusters[df_clusters['cluster'] == 19]['phrase'].tolist()

sample_size = 200

if len(cluster_phrases_full) > sample_size:
    cluster_phrases = random.sample(cluster_phrases_full, sample_size)
    print(f"Randomly sampled {sample_size} phrases dfrom the cluster.")

else:
    cluster_phrases = cluster_phrases_full
    print(f"Cluster has fewer than {sample_size} phrases, using all {len(cluster_phrases_full)} of them.")

print(cluster_phrases[:5])

In [ ]:
import json

formatted_phrases = "\n- ".join(cluster_phrases)

prompt = f"""
You are given a list of phrases.Your Task is to analyze the given list of phrases and extract the main amenities and their specific types mentioned.

You MUST format your response as a single JSON object.

- The keys of the JSON object should be the general amenity category (e.g., "Parking", "Flooring", "Utilities").
- The values should be a list of all the unique, specific types mentioned for that amenity.

For example, if the input phrases were:
- Hardwood floors
- In-unit laundry
- Carpeted bedrooms

Your output should be:
```json
{{
  "Flooring": ["Hardwood", "Carpeted"],
  "Laundry": ["In-Unit"]
}}
- {formatted_phrases}
"""

messages = [
    {
        "role": "system",
        "content": "You are a data extraction specialist working with real estate data. Your task is to analyze the given list of phrases and extract the main amenities and their specific types mentioned and always responds in valid JSON format."
    },
    {
        "role": "user",
        "content":prompt
    }
]

try:
    response = text_gen_pipeline(messages, max_new_tokens=1000)
    generated_text = response[0]["generated_text"][-1]['content']

    if generated_text.startswith("```json"):
        generated_text = generated_text[len("```json\n"):]
    if generated_text.endswith("```"):
        generated_text = generated_text[:-len("\n```")]

    amenity_data = json.loads(generated_text)

    print("--- Successfully Extracted Amenities ---")
    print(amenity_data)

    #df_amenities = pd.DataFrame(dict([ (k, pd.Series(v)) for k,v in amenity_data.item() ]))

except (json.JSONDecoderError, IndexError):
    print("--- Failed to parse JSON from the model's response ---")
    print(response_text)


In [ ]:
df_amenities = pd.DataFrame(dict([ (k, pd.Series(v)) for k,v in amenity_data.items() ]))

In [ ]:
df_amenities

In [ ]:
file_path = '/kaggle/working/amenities.csv'
df_amenities.to_csv(file_path, index=False)

In [19]:
import random 

cluster_phrases_full = df_clusters[df_clusters['cluster'] == 19]['phrase'].tolist()

sample_size = 300

if len(cluster_phrases_full) > sample_size:
    cluster_phrases = random.sample(cluster_phrases_full, sample_size)
    print(f"Randomly sampled {sample_size} phrases dfrom the cluster.")

else:
    cluster_phrases = cluster_phrases_full
    print(f"Cluster has fewer than {sample_size} phrases, using all {len(cluster_phrases_full)} of them.")

print(cluster_phrases[:5])

Randomly sampled 300 phrases dfrom the cluster.
['Apartment features include: Air conditioner, On Bus Line, Dishwasher, Furnished, In-Unit Laundry, Balcony, Deck, Patio, Sheltered parking, elevator in building', 'Apartment available amenities: Pool, Surface Parking, On Bus Line, Balcony, Deck, Patio, Dishwasher, Garbage Disposal, Refrigerator, On-Site Laundry', 'Apartment features include: Dishwasher, Furnished, Fitness facilities', 'A/c- Balcony, Deck, Patio', 'Apartment features include: Fireplace- Furnished- Balcony, Deck, Patio- On-Site Laundry- Student- Pool- Dishwasher- A/c']


In [20]:
import json

formatted_phrases = "\n- ".join(cluster_phrases)

prompt = f"""
You are an expert real estate analyst. Your task is to analyze a list of phrases and summarize its contents. The phrases come from a single cluster of related terms.

Here is the list of phrases:
- {formatted_phrases}

Based on this list, please provide the following:
1.  **Main Theme:** A short, descriptive title for this category of features (e.g., "Property Addresses", "Rental Rates", "Unit Amenities").
2.  **Key Features:** A clear, bulleted list of the specific types of information mentioned.
"""

messages = [
    {
        "role": "user",
        "content":prompt
    }
]

response = text_gen_pipeline(messages, max_new_tokens=1000)
generated_text = response[0]["generated_text"][-1]['content']
print(generated_text)



Okay, here’s an analysis of the provided apartment features and amenities, followed by the requested summaries:

**1. Main Theme:**  **Apartment Amenities & Features** (or potentially “Apartment Features & Amenities – Student Housing Focus”)

**2. Key Features (Bulleted List):**

*   **Appliances:** Dishwasher, Refrigerator, Garbage Disposal, Range/Oven, Microwave, Washer/Dryer Connections
*   **Heating/Cooling:** Air Conditioner (A/c), Forced Air, Central A/C, Overhead Fans
*   **Parking:** Surface Parking, Sheltered Parking, Garage - Attached
*   **Outdoor Spaces:** Balcony, Deck, Patio
*   **Storage:** Storage, Walk-in Closets
*   **Laundry:** In-Unit Laundry, On-Site Laundry
*   **Community Amenities:** Pool, Fitness Facilities
*   **Transportation:** On Bus Line, Public Transportation
*   **Accessibility:** Handicapped Access
*   **Living Spaces:** Living Room, Den
*   **Security:** Controlled Access
*   **Other Features:** Fireplace, Wood Floors, Carpet, Trash Removal Included




In [9]:
import pandas as pd

# Assume 'rental_data' is the Python dictionary parsed from the model's JSON response
# Example rental_data:
# rental_data = {
#   "amenities": {
#     "Flooring": ["Hardwood", "Carpeted"],
#     "Laundry": ["In-Unit"],
#     "Parking": ["Sheltered"]
#   },
#   "unit_specifications": {
#     "types": ["Studio", "1 Bedroom"]
#   }
# }

# --- MODIFIED PART STARTS HERE ---

# 1. Process the 'amenities' section (this is the same as your old code)
amenities_dict = rental_data.get('amenities', {})
df_amenities = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in amenities_dict.items()]))

# 2. Process the 'unit_specifications' section
unit_specs_dict = rental_data.get('unit_specifications', {})
unit_types_list = unit_specs_dict.get('types', [])

# Create a new dictionary to hold the flattened unit types
# This turns ['Studio', '1 Bedroom'] into {'Unit Type 1': 'Studio', 'Unit Type 2': '1 Bedroom'}
unit_types_flat_dict = {f'Unit Type {i+1}': [unit_type] for i, unit_type in enumerate(unit_types_list)}
df_units = pd.DataFrame(unit_types_flat_dict)

# 3. Combine the two DataFrames side-by-side
df_rental = pd.concat([df_amenities, df_units], axis=1)

# --- MODIFIED PART ENDS HERE ---


# The saving part is the same
file_path = '/kaggle/working/rental.csv'
df_rental.to_csv(file_path, index=False)

print(f"Rental data successfully flattened and saved to {file_path}")
display(df_rental)

Rental data successfully flattened and saved to /kaggle/working/rental.csv


,Parking,Flooring,Laundry,Kitchen,Pool,Fitness,Community,Exterior,Other,Unit Type 1,Unit Type 2,Unit Type 3,Unit Type 4,Unit Type 5,Unit Type 6,Unit Type 7,Unit Type 8
0,On-Site Laundry,Hardwood,In-Unit Laundry,Premium Granite Counter Tops,Pool,Fitness Facilities,Community Room,Gated Property,Walk-In Closet,Studio,1 Bedroom,2 Bedroom,3 Bedroom,4 Bedroom,5 Bedroom,1st and 2nd floor units,Townhomes
1,Garage,Tile,Washer & Dryer Connections,Stainless Appliances,Saltwater Swimming Pool,2 Fitness Centers,Club House,Gated Setting,Sun Room,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ample Parking,Laminate Wood,Washer/Dryer Combo,Newer Appliances,Very Large Pool,Fully-Equipped Fitness Room,Playgrounds,Rustic Dark Brown with Stone-like Brick and Ba...,Walk-In Closets in Bedrooms,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Attached Garage,NaN,Washer and Dryers,NaN,NaN,NaN,Picnic Areas with Grills,Tan and Brown Stacked Stone Exteriors,Fireplaces,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Off-Street Parking,NaN,NaN,NaN,NaN,NaN,Public Area Wifi,Bright Orange Entrance,Balcony,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pink Brick with Beige Siding,Deck,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Patio,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Private Patios,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Storage,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Garbage Disposal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
